In [1]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import graphviz
import scipy.stats as stats

trn = pd.read_csv('wdbc_train.csv')
devn = pd.read_csv('wdbc_dev.csv')

features = trn.columns[:-1].tolist()
trn["Diagnosis"] = (trn["Diagnosis"] == "M").astype(int)
devn["Diagnosis"] = (devn["Diagnosis"] == "M").astype(int)
print(features)

x = trn[features].values #extracts all features except Diagnosis column
y = trn["Diagnosis"].values.reshape(-1,1) #target data column, output variable
dx = devn[features].values
dy = devn["Diagnosis"].values.reshape(-1,1) 
#reshape(-1,1) converts the data into n rows and 1 column making it 2D

['Radius', 'Texture', 'Perimeter', 'Area', 'Smoothness', 'Compactness', 'Concavity', 'ConcavePoints', 'Symmetry', 'FractalDimension', 'seRadius', 'seTexture', 'sePerimeter', 'seArea', 'seSmoothness', 'seCompactness', 'seConcavity', 'seConcavePoints', 'seSymmetry', 'seFractalDimension', 'worstRadius', 'worstTexture', 'worstPerimeter', 'worstArea', 'worstSmoothness', 'worstCompactness', 'worstConcavity', 'worstConcavePoints', 'worstSymmetry', 'worstFractalDimension']


In [2]:
class Node():
    def __init__(self, feature = None, threshold = None, ltree = None, rtree = None, ig = None, value = None):
        self.feature = feature
        self.threshold = threshold
        self.ltree = ltree
        self.rtree = rtree
        self.ig = ig #effectiveness of a split
        self.value = value

In [3]:
class DTree():
    def __init__(self, maxdpt = 30, mode = "entropy", chi=0.05):
        self.maxdpt = maxdpt
        self.mode = mode
        self.chi = chi
    def datasplitting(self, dt, ftr, threshold):
        ldt = dt[dt[:,ftr] <= threshold]
        rdt = dt[dt[:,ftr] > threshold]
        return ldt, rdt
    def entropy(self,data):
        entropy = 0
        lbls = np.unique(data)
        for lbl in lbls:
            prob = len(data[data == lbl])/len(data)
            if prob > 0:
                entropy += -prob * np.log2(prob)
            else: 
                entropy = 0
        return entropy
    def igain(self,root,l,r):
        wl = len(l)/len(root)
        wr = len(r)/len(root)
        wentropy = (wl*self.entropy(l))+(wr*self.entropy(r))
        return self.entropy(root)-wentropy
    def gini(self, data):
        gini = 1
        lbls = np.unique(data)
        for lbl in lbls:
            prob = len(data[data==lbl])/len(data)
            gini -= prob**2
        return gini
    def ggain(self, root, l, r):
        wl = len(l)/len(root)
        wr = len(r)/len(root)
        wgini = (wl*self.gini(l))+(wr*self.gini(r))
        return self.gini(root) - wgini

    def chisquare(self, parent, left, right):
        unclass = np.unique(parent)
        size = len(parent)
        obleft = [np.sum(left == cls) for cls in unclass]
        obright = [np.sum(right == cls) for cls in unclass]
        totcount = [np.sum(parent == cls) for cls in unclass]
        exleft = [(count * len(left)) / size for count in totcount]
        exright = [(count * len(right)) / size for count in totcount]
        chisq = 0
        for obsl, obsr, expl, expr in zip(obleft, obright, exleft, exright):
            chisq += ((obsl - expl) ** 2 / expl) if expl > 0 else 0
            chisq += ((obsr - expr) ** 2 / expr) if expr > 0 else 0
        
        return chisq
        
    def leafvalue(self, data):
        data = list(data)
        return max(data, key = data.count)
    def bestsplit(self,dset, nf):
        bs = {'gain': -1, 'feature':None, 'threshold': None}
        for findex in range(nf):
            fval = dset[:,findex]
            tholds = np.unique(fval)
            for threshold in tholds:
                ldata, rdata = self.datasplitting(dset, findex, threshold)
                if len(ldata) and len(rdata):
                    root = dset[:,-1]
                    lydata, rydata = ldata[:,-1],rdata[:,-1]
                    if self.mode == "gini":
                        gain = self.ggain(root, lydata, rydata)
                    elif self.mode == "entropy":
                        gain = self.igain(root, lydata, rydata)
                    chival = self.chisquare(root, lydata, rydata)
                    pval = 1 - stats.chi2.cdf(chival, df=len(np.unique(root)) - 1)
                    if pval < self.chi and gain > bs["gain"]:
                        bs["feature"] = findex
                        bs["threshold"] = threshold
                        bs["leftdata"] = ldata
                        bs["rightdata"] = rdata
                        bs["gain"] = gain
        return bs
    def buildtree(self, dset, cdepth=0):
        x, y = dset[:, :-1], dset[:, -1]
        nsamp, nfeatures = x.shape
    
        # Stopping condition: if all labels are the same or max depth reached
        if cdepth >= self.maxdpt or len(np.unique(y)) == 1 or nsamp <= 1:
            leafval = self.leafvalue(y)
            return Node(value=leafval)  # Return a leaf node with the most common class label
        
        # Find the best split
        bs = self.bestsplit(dset, nfeatures)
        if bs["gain"] > 0:
            # Recursively build left and right nodes
            lnode = self.buildtree(bs["leftdata"], cdepth + 1)
            rnode = self.buildtree(bs["rightdata"], cdepth + 1)
            return Node(bs["feature"], bs["threshold"], lnode, rnode, bs["gain"])
    
        # If no valid split found, return a leaf node
        leafval = self.leafvalue(y)
        return Node(value=leafval)
    def fit(self, x, y):
        dset = np.concatenate((x,y.reshape(-1,1)),axis=1)
        self.root = self.buildtree(dset)
    def predict(self, X):
        p = [self.makeprediction(x, self.root) for x in X]
        return np.array(p)
    def makeprediction(self, x, node):
        if isinstance(node, Node) and node.value is not None:
            return node.value
        attribute = x[node.feature]
        if attribute <= node.threshold:
            return self.makeprediction(x, node.ltree)
        return self.makeprediction(x, node.rtree)
    def treeviz(self, node = None, d = None):
        if d is None:
            d = graphviz.Digraph(comment='Decision Tree')
        if node is None:
            node = self.root
            print(self.root)
        if node.value is not None:
            d.node(str(id(node)), f"Class: {node.value}", shape='box')
        else:
            d.node(str(id(node)), f"Feature {node.feature} <= {node.threshold}")
            if node.ltree:
                d.edge(str(id(node)), str(id(node.ltree)), label="True")
                self.treeviz(node.ltree, d)
            if node.rtree:
                d.edge(str(id(node)), str(id(node.rtree)), label="False")
                self.treeviz(node.rtree, d)
        return d
                   

In [4]:
def accuracy(y, ypredict):
    y = y.flatten()
    totalsamp = len(y)
    truepred = np.sum(y == ypredict)
    return (truepred/totalsamp)

def fmetrics(y, ypredict):
    ypredict = np.array(ypredict)
    y = y.flatten()
    nc = len(np.unique(y))
    sensitivity = []
    specificity = []
    for i in range(nc):
        mt = y == i
        mp = ypredict == i
        tpositive = np.sum(mt&mp)
        fpositive = np.sum((mt != True)&mp)
        tnegative = np.sum((mt != True)&(mp != True))
        fnegative = np.sum(mt&(mp != True))
        se = tpositive/(tpositive+fnegative)
        sp = tnegative/(tnegative+fpositive)
        precision = tpositive/(tpositive+fpositive)
        f1 = 2*(precision*se)/(precision*se)
        fpr = fpositive/(fpositive+tnegative)
        fnr = fnegative/(fnegative+tpositive)
        npv = tnegative/(tnegative+fnegative)
        confusionmatrix = np.array([[tpositive,fnegative],[fpositive,tnegative]])
        print("Confusion Matrxi")
        print(confusionmatrix)
        sensitivity.append(se)
        specificity.append(sp)
    avgsensitivity = np.mean(sensitivity)
    avgspecificity = np.mean(specificity)
    metric = (avgsensitivity+avgspecificity)/nc
    return metric, se, sp, precision, f1, fpr, fnr, npv

In [5]:
mode = input("Gini or Entropy")
tree = DTree(7,mode)
tree.fit(x,y)
d = tree.treeviz(tree.root)
d.render("DecisionTree.gv",view = True)

p = tree.predict(dx)

print(f"Model's Accuracy: {accuracy(dy, p)}")
print(f"Model's Accuracy:")
me, se, sp, precision, f1, fpr, fnr, nvp = fmetrics(dy, p)
print(me)
print(f"Recall: {se}")
print(f"Specificity: {sp}")
print(f"Precision: {precision}")
print(f"F1 Score: {f1}")
print(f"FPR Score: {fpr}")
print(f"FNR Score: {fnr}")
print(f"NVP Score: {nvp}")

Gini or Entropy gini


Model's Accuracy: 0.9736842105263158
Model's Accuracy:
Confusion Matrxi
[[71  0]
 [ 3 40]]
Confusion Matrxi
[[40  3]
 [ 0 71]]
0.9651162790697674
Recall: 0.9302325581395349
Specificity: 1.0
Precision: 1.0
F1 Score: 2.0
FPR Score: 0.0
FNR Score: 0.06976744186046512
NVP Score: 0.9594594594594594


In [6]:
def parametertune(xtrain, ytrain, xtest, ytest):
    max_depths = [3, 5, 7, 15, 20, 25]
    modes = ['gini', 'entropy']
    bestp = {}
    besta = 0
    for maxd in max_depths:
        for mode in modes:
            tree = DTree(maxdpt=maxd, mode=mode)
            tree.fit(xtrain, ytrain)
            ypred = tree.predict(xtest)
            curraccuracy = accuracy(ytest, ypred)
            if curraccuracy > besta:
                bestp = {
                    'max_depth': maxd,
                    'mode': mode
                }
                besta = curraccuracy
            print(f"Params: max_depth={maxd}, mode={mode}, Accuracy={curraccuracy}")

    print("\nBest:")
    print(bestp)
    print(f"Best Accuracy: {besta}")

parametertune(x, y, dx, dy)

)07=                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             Params: max_depth=3, mode=gini, Accuracy=0.9385964912280702
                        

Error: no "view" rule for type "application/pdf" passed its test case
       (for more information, add "--debug=1" on the command line)


>Params: max_depth=3, mode=entropy, Accuracy=0.9736842105263158                         Getting file://localhost/home/stu1/s7/sp7289/PS1-Pasupuleti/DecisionTree.gv.pdf 
Params: max_depth=5, mode=gini, Accuracy=0.9736842105263158
Params: max_depth=5, mode=entropy, Accuracy=0.9649122807017544
Params: max_depth=7, mode=gini, Accuracy=0.9736842105263158
Params: max_depth=7, mode=entropy, Accuracy=0.9649122807017544
Params: max_depth=15, mode=gini, Accuracy=0.9649122807017544
Params: max_depth=15, mode=entropy, Accuracy=0.9649122807017544
Params: max_depth=20, mode=gini, Accuracy=0.9649122807017544
Params: max_depth=20, mode=entropy, Accuracy=0.9649122807017544
Params: max_depth=25, mode=gini, Accuracy=0.9649122807017544
Params: max_depth=25, mode=entropy, Accuracy=0.9649122807017544

Best:
{'max_depth': 3, 'mode': 'entropy'}
Best Accuracy: 0.9736842105263158


In [7]:
test = pd.read_csv('wdbc_test.csv')
test["Diagnosis"] = (test["Diagnosis"] == "M").astype(int)
print(features)

xtest = test[features].values
ytest = test["Diagnosis"].values.reshape(-1,1) 
#reshape(-1,1) converts the data into n rows and 1 column making it 2D

['Radius', 'Texture', 'Perimeter', 'Area', 'Smoothness', 'Compactness', 'Concavity', 'ConcavePoints', 'Symmetry', 'FractalDimension', 'seRadius', 'seTexture', 'sePerimeter', 'seArea', 'seSmoothness', 'seCompactness', 'seConcavity', 'seConcavePoints', 'seSymmetry', 'seFractalDimension', 'worstRadius', 'worstTexture', 'worstPerimeter', 'worstArea', 'worstSmoothness', 'worstCompactness', 'worstConcavity', 'worstConcavePoints', 'worstSymmetry', 'worstFractalDimension']


In [8]:
mode = input("Gini or Entropy")
tree = DTree(7,mode)
tree.fit(xtest,ytest)
d = tree.treeviz(tree.root)
d.render("DecisionTree.gv",view = True)

p = tree.predict(xtest)

print(f"Model's Accuracy: {accuracy(ytest, p)}")
print(f"Model's Accuracy:")
me, se, sp, precision, f1, fpr, fnr, nvp = fmetrics(ytest, p)
print(me)
print(f"Recall: {se}")
print(f"Specificity: {sp}")
print(f"Precision: {precision}")
print(f"F1 Score: {f1}")
print(f"FPR Score: {fpr}")
print(f"FNR Score: {fnr}")
print(f"NVP Score: {nvp}")

Gini or Entropy gini


Model's Accuracy: 1.0
Model's Accuracy:
Confusion Matrxi
[[72  0]
 [ 0 42]]
Confusion Matrxi
[[42  0]
 [ 0 72]]
1.0
Recall: 1.0
Specificity: 1.0
Precision: 1.0
F1 Score: 2.0
FPR Score: 0.0
FNR Score: 0.0
NVP Score: 1.0


In [12]:
d = pd.read_csv('wdbc_train_raw.csv')
fc = d.columns[:30]
nb = 6
for c in fc:
    d[c + '_bin'] = pd.qcut(d[c], q=nb, labels=False)
print(d.head())

   Radius  Texture  Perimeter    Area  Smoothness  Compactness  Concavity  \
0   20.57    17.77     132.90  1326.0     0.08474      0.07864     0.0869   
1   19.69    21.25     130.00  1203.0     0.10960      0.15990     0.1974   
2   11.42    20.38      77.58   386.1     0.14250      0.28390     0.2414   
3   18.25    19.98     119.60  1040.0     0.09463      0.10900     0.1127   
4   13.00    21.82      87.50   519.8     0.12730      0.19320     0.1859   

   ConcavePoints  Symmetry  FractalDimension  ...  worstRadius_bin  \
0        0.07017    0.1812           0.05667  ...                5   
1        0.12790    0.2069           0.05999  ...                5   
2        0.10520    0.2597           0.09744  ...                2   
3        0.07400    0.1794           0.05742  ...                5   
4        0.09353    0.2350           0.07389  ...                3   

   worstTexture_bin  worstPerimeter_bin  worstArea_bin  worstSmoothness_bin  \
0                 2                  